In [9]:
import requests
import pandas as pd
import json
import os

# Configurações da API Local
BASE_URL = "http://localhost:3003"
API_KEY = "UgdRwHpekKRWafd+gA0Q1I72iCeQuKAqArrMEkU43f6UdgOJDsUnqoMKDB+2hhk8st9DNW9gSozSUdvKGI2w89RMMlMGEAEb1hYILnjroouAAMvAvZygGORoaX3DYQi4Dj8KX0mtHRcYRS7XW76BOxpijFVUa3IdhYLhZJIL1uo="
QUANTITY = 10000
HEADERS = {
    "x-api-key": API_KEY,
    "Content-Type": "application/json"
}
ENDPOINT = f"{BASE_URL}/processos-por-documento"
SCRIPT_NAME = "processos-por-cpf"

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [10]:
input_path = os.path.join("..", "input", "processos_documentos.json")

with open(input_path, "r", encoding="utf-8") as f:
    document_list = json.load(f)

print(f"Total de documentos a processar: {len(document_list)}")

Total de documentos a processar: 383


In [11]:
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import re

# Cores pastéis por grupo de coluna: (header_hex, data_hex)
# Amarelo    → identidade (Nome, CPF)
# Azul       → identificação do processo (Número, Tribunal, etc.)
# Laranja    → classificação jurídica (Área, Classe, Assuntos, Ramo)
# Verde      → situação e tempo (Status, Data)
# Roxo       → financeiro (Valor da Causa)
# Azul-claro → administrativo (Juiz, Partes, URL)
COLORS_COM = {
    "Nome":                 ("FFD966", "FFFCE8"),
    "CPF Consultado":       ("FFD966", "FFFCE8"),
    "Número do Processo":   ("9DC3E6", "EBF3FB"),
    "Tribunal":             ("9DC3E6", "EBF3FB"),
    "Órgão Julgador":       ("9DC3E6", "EBF3FB"),
    "Instância":            ("9DC3E6", "EBF3FB"),
    "Sistema":              ("9DC3E6", "EBF3FB"),
    "Área":                 ("F4B183", "FEF3EA"),
    "Segmento":             ("F4B183", "FEF3EA"),
    "Ramo do Direito":      ("F4B183", "FEF3EA"),
    "Classe":               ("F4B183", "FEF3EA"),
    "Assuntos Principais":  ("F4B183", "FEF3EA"),
    "Status":               ("A9D18E", "EEF5E9"),
    "Data de Distribuição": ("A9D18E", "EEF5E9"),
    "Valor da Causa (R$)":  ("C9A0DC", "F5EEF8"),
    "Juiz":                 ("BDD7EE", "F0F6FC"),
    "Total de Partes":      ("BDD7EE", "F0F6FC"),
    "Partes":               ("BDD7EE", "F0F6FC"),
    "URL do Processo":      ("C0C0C0", "F5F5F5"),
}

COLORS_SEM = {
    "Nome":           ("FFD966", "FFFCE8"),
    "CPF Consultado": ("FFD966", "FFFCE8"),
}

THIN_BORDER = Border(
    left=Side(style="thin", color="D0D0D0"),
    right=Side(style="thin", color="D0D0D0"),
    top=Side(style="thin", color="D0D0D0"),
    bottom=Side(style="thin", color="D0D0D0"),
)

CACHE_DIR = os.path.join("..", "responses", SCRIPT_NAME, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)


def normalize_doc(document):
    """Remove tudo que não for dígito — aceita 000.000.000-00 ou 00000000000."""
    return re.sub(r"\D", "", document)


def format_sheet(ws, df, col_colors):
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    data_align = Alignment(horizontal="left", vertical="center", wrap_text=False)
    money_align = Alignment(horizontal="right", vertical="center")

    for col_idx, col_name in enumerate(df.columns, start=1):
        header_hex, data_hex = col_colors.get(col_name, ("D9D9D9", "F5F5F5"))
        header_fill = PatternFill("solid", fgColor=header_hex)
        data_fill = PatternFill("solid", fgColor=data_hex)

        header_cell = ws.cell(row=1, column=col_idx)
        header_cell.fill = header_fill
        header_cell.font = Font(bold=True, color="3B3B3B", size=10)
        header_cell.alignment = header_align
        header_cell.border = THIN_BORDER

        is_money = col_name == "Valor da Causa (R$)"
        for row_idx in range(2, ws.max_row + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.fill = data_fill
            cell.border = THIN_BORDER
            cell.font = Font(size=10)
            if is_money:
                cell.number_format = '"R$" #,##0.00'
                cell.alignment = money_align
            else:
                cell.alignment = data_align

    ws.row_dimensions[1].height = 32
    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = get_column_letter(col_idx)
        max_content = max(
            len(str(col_name)),
            max(
                (len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(2, ws.max_row + 1)),
                default=0,
            ),
        )
        ws.column_dimensions[col_letter].width = min(max_content + 3, 55)

    ws.freeze_panes = "A2"


def fetch_processos(document):
    doc_key = normalize_doc(document)
    cache_file = os.path.join(CACHE_DIR, f"{doc_key}.json")

    if os.path.exists(cache_file):
        print(f"  -> Cache encontrado, pulando chamada à API")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    body = {
        "identifier": 1,
        "document": doc_key,
        "webhook_url": "https://webhook.site/c1a70033-31b9-4879-a4e2-6f7ffbb7bd40",
        "async": False,
        "quantity": QUANTITY,
    }
    try:
        response = requests.post(ENDPOINT, headers=HEADERS, json=body)
        response.raise_for_status()
        payload = response.json()
        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        return payload
    except requests.exceptions.RequestException as e:
        print(f"  -> Erro na chamada da API: {e}")
        return None


def extract_rows(nome, documento, payload):
    rows = []
    for proc in payload.get("data", []):
        assuntos_principais = "; ".join(
            a["titulo"] for a in proc.get("assuntos_cnj", []) if a.get("e_principal")
        )
        valor_causa = proc.get("valor_causa") or {}
        classe = proc.get("classe") or {}

        partes_str = " | ".join(
            f"Nome: {p.get('nome', 'N/A')}, Doc: {p.get('cpf') or p.get('cnpj') or 'N/A'}"
            for p in proc.get("partes", [])
        )

        rows.append({
            "Nome":                 nome,
            "CPF Consultado":       documento,
            "Número do Processo":   proc.get("numero_processo_unico"),
            "Tribunal":             proc.get("tribunal"),
            "Órgão Julgador":       proc.get("orgao_julgador"),
            "Área":                 proc.get("area"),
            "Segmento":             proc.get("segmento"),
            "Sistema":              proc.get("sistema"),
            "Instância":            proc.get("instancia"),
            "Status":               proc.get("status"),
            "Ramo do Direito":      proc.get("ramo_direito"),
            "Classe":               classe.get("nome"),
            "Assuntos Principais":  assuntos_principais or "N/A",
            "Data de Distribuição": proc.get("data_distribuicao"),
            "Valor da Causa (R$)":  valor_causa.get("valor"),
            "Juiz":                 proc.get("juiz"),
            "Total de Partes":      proc.get("total_partes"),
            "Partes":               partes_str or "N/A",
            "URL do Processo":      proc.get("url_processo"),
        })
    return rows


print(f"Funções e esquema de cores definidos. Cache em: {CACHE_DIR}")

Funções e esquema de cores definidos. Cache em: ../responses/processos-por-cpf/cache


In [12]:
com_processos = []
sem_processos = []
erros = []

for entry in document_list:
    nome = entry.get("nome", "")
    documento = entry.get("documento", "")
    print(f"Processando: {nome} ({documento})")

    payload = fetch_processos(documento)

    if payload is None:
        erros.append({"Nome": nome, "CPF Consultado": documento})
        continue

    processos = payload.get("data", [])
    if not processos:
        print(f"  -> Sem processos encontrados")
        sem_processos.append({"Nome": nome, "CPF Consultado": documento})
    else:
        print(f"  -> {len(processos)} processo(s) encontrado(s)")
        com_processos.extend(extract_rows(nome, documento, payload))

print("\n--- Gerando Arquivo Final ---")

df_com = pd.DataFrame(com_processos) if com_processos else pd.DataFrame(
    columns=["Nome", "CPF Consultado", "Número do Processo", "Tribunal",
             "Órgão Julgador", "Área", "Segmento", "Sistema", "Instância",
             "Status", "Ramo do Direito", "Classe", "Assuntos Principais",
             "Data de Distribuição", "Valor da Causa (R$)", "Juiz",
             "Total de Partes", "Partes", "URL do Processo"]
)
df_sem = pd.DataFrame(sem_processos) if sem_processos else pd.DataFrame(
    columns=["Nome", "CPF Consultado"]
)

output_dir = os.path.join("..", "responses", SCRIPT_NAME)
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "relatorio_processos.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_com.to_excel(writer, sheet_name="Com Processos", index=False)
    df_sem.to_excel(writer, sheet_name="Sem Processos", index=False)
    format_sheet(writer.sheets["Com Processos"], df_com, COLORS_COM)
    format_sheet(writer.sheets["Sem Processos"], df_sem, COLORS_SEM)

# Resumo final
total_buscados = len(document_list)
total_com = len(set(r["CPF Consultado"] for r in com_processos))
total_sem = len(sem_processos)
total_erros = len(erros)

print(f"\n{'='*50}")
print(f"RESUMO DA EXECUÇÃO")
print(f"{'='*50}")
print(f"Total de documentos buscados : {total_buscados}")
print(f"Com processos                : {total_com}")
print(f"Sem processos                : {total_sem}")
print(f"Erros na API                 : {total_erros}")
if erros:
    print("\nDocumentos com erro:")
    for e in erros:
        print(f"  - {e['Nome']} ({e['CPF Consultado']})")
print(f"{'='*50}")
print(f"\nArquivo salvo em: {output_file}")

if not df_com.empty:
    print("\nPrévia - Com Processos:")
    display(df_com.head())

if not df_sem.empty:
    print("\nPrévia - Sem Processos:")
    display(df_sem.head())

Processando: Raquel Elisabet Vita de Draghi (00495108782)
  -> 3 processo(s) encontrado(s)
Processando: Maria Seixas (65168151868)
  -> 18 processo(s) encontrado(s)
Processando: Leandro Rubio (10871428733)
  -> 21 processo(s) encontrado(s)
Processando: Eduardo Ferreira Santos (57326320853)
  -> 7 processo(s) encontrado(s)
Processando: Edison Honório (66685613815)
  -> 27 processo(s) encontrado(s)
Processando: Daniele Q. Fucciolo Penalva (21441817808)
  -> 2 processo(s) encontrado(s)
Processando: Glenia Azevedo Zerbone (98403192720)
  -> 9 processo(s) encontrado(s)
Processando: Brenda Kimberly Rodrigues (12555727604)
  -> 1 processo(s) encontrado(s)
Processando: Carolina Codicasa (38747450827)
  -> Sem processos encontrados
Processando: André Ibrahim David (13440542858)
  -> 4 processo(s) encontrado(s)
Processando: Maria Fernanda Ribeiro Gomes Siqueira (27147449827)
  -> 2 processo(s) encontrado(s)
Processando: Graziela Cunha da Costa (57205817315)
  -> 10 processo(s) encontrado(s)
Proc

,Nome,CPF Consultado,Número do Processo,Tribunal,Órgão Julgador,Área,Segmento,Sistema,Instância,Status,Ramo do Direito,Classe,Assuntos Principais,Data de Distribuição,Valor da Causa (R$),Juiz,Total de Partes,Partes,URL do Processo
0,Raquel Elisabet Vita de Draghi,00495108782,50135123120258130525,TJ-MG,1ª VARA CIVEL DA COMARCA DE POUSO ALEGRE,NaN,JUSTICA ESTADUAL,PJE-TJMG,1,EM TRAMITACAO,DIREITO TRIBUTARIO,EMBARGOS A EXECUCAO FISCAL,DIREITO TRIBUTARIO,2025-07-21T00:00:00,4116.20,JOSE HELIO DA SILVA,2,"Nome: RAQUEL ELISABET VITA DE DRAGHI, Doc: 004...",https://pje-consulta-publica.tjmg.jus.br/pje/C...
1,Raquel Elisabet Vita de Draghi,00495108782,50060208520258130525,TJ-MG,1ª VARA CIVEL DA COMARCA DE POUSO ALEGRE,NaN,JUSTICA ESTADUAL,PJE-TJMG,1,EM TRAMITACAO,DIREITO TRIBUTARIO,EXECUCAO FISCAL,DIREITO TRIBUTARIO,2025-04-02T00:00:00,4116.20,JOSE HELIO DA SILVA,2,"Nome: RAQUEL ELISABET VITA DE DRAGHI, Doc: 004...",https://pje-consulta-publica.tjmg.jus.br/pje/C...
2,Raquel Elisabet Vita de Draghi,00495108782,50068846520218130525,TJ-MG,3ª VARA CIVEL DA COMARCA DE POUSO ALEGRE,NaN,JUSTICA ESTADUAL,PJE-TJMG,1,ARQUIVAMENTO DEFINITIVO,DIREITO TRIBUTARIO,EXECUCAO FISCAL,ISS/ IMPOSTO SOBRE SERVICOS,2021-08-12T00:00:00,5376.27,JULIANA MENDES PEDROSA,2,"Nome: RAQUEL ELISABET VITA DE DRAGHI, Doc: 004...",https://pje-consulta-publica.tjmg.jus.br/pje/C...
3,Maria Seixas,65168151868,10016192420225020089,TST,GABINETE DO MINISTRO LUIZ JOSE DEZENA DA SILVA,TRABALHISTA,JUSTICA DO TRABALHO,PJE-TST,3,EM TRAMITACAO,DIREITO DO TRABALHO,AGRAVO DE INSTRUMENTO EM RECURSO DE REVISTA,SENTENCA DE LIQUIDACAO,2024-08-30T17:30:13,12331.71,NaN,3,"Nome: MARIA CONCEICAO SEIXAS DA SILVA, Doc: 65...",https://pje.tst.jus.br/consultaprocessual/deta...
4,Maria Seixas,65168151868,10016192420225020089,TRT-2,14ª TURMA - CADEIRA 3,TRABALHISTA,JUSTICA DO TRABALHO,PJE-TRT2,2,ARQUIVAMENTO DEFINITIVO,DIREITO DO TRABALHO,AGRAVO DE PETICAO,SENTENCA DE LIQUIDACAO,2023-04-04T15:22:56,12331.71,NaN,4,"Nome: MARIA CONCEICAO SEIXAS DA SILVA, Doc: 65...",https://pje.trt2.jus.br/consultaprocessual/det...



Prévia - Sem Processos:


,Nome,CPF Consultado
0,Carolina Codicasa,38747450827
1,Juliana Araujo Verrenja,45459653851
2,Caroline de Araújo,08297358950
3,Daniela Cristina Lima de Oliveira,43022105835
4,Jessica Nere Silva,37649170837
